# Pipeline MLCQ — Visão geral dos dados

Este notebook resume o fluxo: **CSV original** → **busca do código nos repositórios** → **CSV com snippets** → **geração de SQL para a base**.

## 1. CSV original

Carregamento e visão geral do dataset original (metadados: repo, commit, path, start_line, end_line — **sem trecho de código**).

In [41]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd
if not (repo_root / "datasets" / "mlcq_original_dataset.csv").exists():
    repo_root = repo_root.parent.parent
BASE = repo_root / "datasets"
original = pd.read_csv(BASE / "mlcq_original_dataset.csv", sep=";")
original.head()

,id,reviewer_id,sample_id,smell,severity,review_timestamp,type,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project
0,526,6,5771277,feature envy,none,2019-03-27 10:34:53.041496,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1
1,527,6,5771277,long method,none,2019-03-27 10:34:53.042443,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1
2,528,6,5786929,blob,critical,2019-03-27 10:37:38.107923,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1
3,529,6,5786929,data class,critical,2019-03-27 10:37:38.109068,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1
4,530,6,5788107,feature envy,none,2019-03-27 10:37:49.627100,function,org.apache.tika.parser.ocr.TesseractOCRConfig#...,git@github.com:apache/tika.git,4131c6e30f2e0eb1feb85e0f7576531d4e830468,/tika-parsers/src/main/java/org/apache/tika/pa...,531,534,https://github.com/apache/tika/blob/4131c6e30f...,1


In [42]:
original.shape

(14739, 15)

*Pre-processing of dataset*

In [43]:
TARGET_SMELLS = ["feature envy", "long method", "data class"]
RELEVANT_SEVERITIES = ["major", "critical"]
SEVERITY_PRIORITY = {"critical": 0, "major": 1}

# Filter by relevant projects and copy dataset to perform changes
original = original[original['is_from_industry_relevant_project'] == '1'].copy()
# Convert lines to numeric where suitable
original['start_line'] = pd.to_numeric(original['start_line'], errors='coerce')
original['end_line'] = pd.to_numeric(original['end_line'], errors='coerce')
original = original.dropna(subset=['start_line', 'end_line'])
original['start_line'] = original['start_line'].astype(int)
original['end_line'] = original['end_line'].astype(int)
# Remove lines with empty core data
original = original.dropna(subset=['repository', 'commit_hash', 'path'])
# Keep only the smells we evaluate, and only severities that matter
original = original[
    original['smell'].isin(TARGET_SMELLS)
    & original['severity'].isin(RELEVANT_SEVERITIES)
]
# Dedup: 1 anotação por (repo, commit, path, smell). Critical > major em caso de divergência
original = original.assign(_sev_rank=original['severity'].map(SEVERITY_PRIORITY))
original = (
    original.sort_values(['_sev_rank', 'id'])
    .drop_duplicates(subset=['repository', 'commit_hash', 'path', 'smell'], keep='first')
    .drop(columns='_sev_rank')
)
# Descomente abaixo para salvar o dataset filtrado (uma linha por anotação)
# original.to_csv(BASE / "mlcq_smell_occurrences.csv", sep=";", index=False)

In [44]:
original.shape

(524, 15)

## 2. Busca do código nos repositórios

O dataset **não traz o conteúdo dos arquivos**, apenas as indicações (repositório, commit, arquivo). Por isso é necessário buscar via API do GitHub. `collect_mlcq_samples_with_source_code.process_dataset` lê `mlcq_smell_occurrences.csv`, agrupa por arquivo (mesma lógica do passo 4 — arrays de `smell_occurrence_ids`/`annotated_smells`/`severities`/`max_severity`/`annotation_count`), busca `file_content` uma única vez por arquivo e salva direto em `mlcq_files.csv`. Não há mais `code_snippet` (o LLM consome o arquivo inteiro) nem CSV intermediário `mlcq_enriched_dataset.csv`.

Mantida comentada para não reprocessar.

In [ ]:
# Descomente abaixo para refazer a busca do source code nos repositórios.
#
# Fluxo novo: lê mlcq_smell_occurrences.csv, deduplica por arquivo agregando
# smell_occurrence_ids/annotated_smells/severities/max_severity/annotation_count,
# busca file_content uma vez por arquivo, e salva mlcq_files.csv final.
# Requer process_dataset adaptado pro schema agregado (sem code_snippet, sem split manual).
#
# import sys
# sys.path.insert(0, str(BASE))
# from collect_mlcq_samples_with_source_code import process_dataset
#
# process_dataset(
#     original_csv_file=str(BASE / "mlcq_smell_occurrences.csv"),
#     json_file=str(BASE / "mlcq_files.json"),
#     csv_file_with_source_code=str(BASE / "mlcq_files.csv"),
# )

## 3. CSV consolidado por arquivo

`mlcq_files.csv` é o artefato final que o restante do pipeline consome — uma linha por arquivo com `file_content` e os arrays `smell_occurrence_ids`/`annotated_smells`/`severities`/`max_severity`/`annotation_count`. Ele pode ser produzido por:

- **Fluxo novo** (passo 2): `process_dataset` agrupa e busca direto a partir de `mlcq_smell_occurrences.csv`.
- **Bridge legado** (passo 4 abaixo): a partir do `mlcq_enriched_dataset.csv` que já existe em disco. Rode o passo 4 primeiro se você ainda não tem o `mlcq_files.csv`.

In [9]:
files = pd.read_csv(BASE / "mlcq_files.csv", sep=";")
files.head()

,repo_url,commit_hash,file_path,file_content,annotated_smells,severities,annotation_count,smell_occurrence_ids,max_severity
0,git@github.com:Esri/geoportal-server-harvester...,b8c69260e3d6ec10df6514c201219e690cfba048,/geoportal-connectors/geoportal-harvester-waf/...,"/*\n * Copyright 2016 Esri, Inc..\n *\n * Lice...",['data class'],['major'],2,"[953, 10810]",major
1,git@github.com:Esri/maps-app-android.git,1af1f74ece08f678ce7de7bf173034d30e1cb100,/maps-app/src/main/java/com/esri/android/mapsa...,/* Copyright 1995-2016 Esri\n *\n * Licensed u...,['long method'],['major'],1,[11092],major
2,git@github.com:IBM/ibm-cos-sdk-java.git,d6b03864c15c622ce439e39f20ab41a77dc1cc83,/ibm-cos-java-sdk-kms/src/main/java/com/ibm/cl...,"/*\n * Copyright 2012-2017 Amazon.com, Inc. or...",['data class'],['major'],2,"[1068, 12216]",major
3,git@github.com:Microsoft/azure-tools-for-java.git,d121e8ac9cc3ab400e5b49c8b372280ae332f3fb,/PluginsAndFeatures/azure-toolkit-for-intellij...,package com.microsoft.intellij;\n\nimport com....,['data class'],['major'],1,[1125],major
4,git@github.com:Microsoft/azure-tools-for-java.git,d121e8ac9cc3ab400e5b49c8b372280ae332f3fb,/Utils/azuretools-core/src/com/microsoft/azure...,/*\n * Copyright (c) Microsoft Corporation\n *...,['data class'],['major'],2,"[10825, 13474]",major


In [10]:
files.shape

(436, 9)

## 4. Bridge legado — consolidação a partir do `mlcq_enriched_dataset.csv`

Lê o `mlcq_enriched_dataset.csv` (que tem uma linha por anotação com `file_content`), filtra `blob` e agrega por arquivo, salvando o `mlcq_files.csv` que o passo 3 consome. **Só é necessário rodar enquanto o fluxo novo (passo 2) não foi executado** — depois do refresh esse passo vira redundante.

In [45]:
TARGET_SMELLS = ["feature envy", "long method", "data class"]
RELEVANT_SEVERITIES = ["major", "critical"]
SEVERITY_PRIORITY = {"critical": 0, "major": 1}

enriched = pd.read_csv(BASE / "mlcq_enriched_dataset.csv")
filtered = enriched[
    enriched["smell"].isin(TARGET_SMELLS)
    & enriched["severity"].isin(RELEVANT_SEVERITIES)
].copy()
# Dedup: 1 anotação por (repo, commit, path, smell). Critical > major em caso de divergência
filtered = filtered.assign(_sev_rank=filtered["severity"].map(SEVERITY_PRIORITY))
filtered = (
    filtered.sort_values(["_sev_rank", "id"])
    .drop_duplicates(subset=["repo_url", "commit_hash", "file_path", "smell"], keep="first")
    .drop(columns="_sev_rank")
)

files = (
    filtered
    .groupby(["repo_url", "commit_hash", "file_path"], as_index=False)
    .agg(
        file_content=("file_content", "first"),
        annotated_smells=("smell", lambda s: sorted(set(s))),
        annotation_count=("id", "count"),
        smell_occurrence_ids=("id", lambda s: sorted(s.tolist())),
    )
)

print(f"Annotações major/critical (após dedup): {len(filtered)}")
print(f"Arquivos únicos:                       {len(files)}")

files.to_csv(BASE / "mlcq_files.csv", sep=";", index=False)
print(f"Saved: mlcq_files.csv")

files.head()

Annotações major/critical (após dedup): 476
Arquivos únicos:                       436
Saved: mlcq_files.csv


,repo_url,commit_hash,file_path,file_content,annotated_smells,annotation_count,smell_occurrence_ids
0,git@github.com:Esri/geoportal-server-harvester...,b8c69260e3d6ec10df6514c201219e690cfba048,/geoportal-connectors/geoportal-harvester-waf/...,"/*\n * Copyright 2016 Esri, Inc..\n *\n * Lice...",[data class],1,[953]
1,git@github.com:Esri/maps-app-android.git,1af1f74ece08f678ce7de7bf173034d30e1cb100,/maps-app/src/main/java/com/esri/android/mapsa...,/* Copyright 1995-2016 Esri\n *\n * Licensed u...,[long method],1,[11092]
2,git@github.com:IBM/ibm-cos-sdk-java.git,d6b03864c15c622ce439e39f20ab41a77dc1cc83,/ibm-cos-java-sdk-kms/src/main/java/com/ibm/cl...,"/*\n * Copyright 2012-2017 Amazon.com, Inc. or...",[data class],1,[1068]
3,git@github.com:Microsoft/azure-tools-for-java.git,d121e8ac9cc3ab400e5b49c8b372280ae332f3fb,/PluginsAndFeatures/azure-toolkit-for-intellij...,package com.microsoft.intellij;\n\nimport com....,[data class],1,[1125]
4,git@github.com:Microsoft/azure-tools-for-java.git,d121e8ac9cc3ab400e5b49c8b372280ae332f3fb,/Utils/azuretools-core/src/com/microsoft/azure...,/*\n * Copyright (c) Microsoft Corporation\n *...,[data class],1,[10825]


## 5. Gerar SQL para a base de dados (opcional)

Gera os `.sql` de inserts em `datasets/`, úteis como artefato portátil (aplicar via `psql` em outro ambiente, debug, etc.). Pra alimentar a DB local **não é necessário** — o passo 6 carrega direto do CSV via `psycopg2`.

In [46]:
# Descomente abaixo para gerar o arquivo de inserts

import sys
sys.path.insert(0, str(BASE))
from generate_sql_inserts_from_csv import (
    generate_sql_inserts,
    generate_sql_inserts_for_files,
)

# Inserts para mlcq_smell_occurrences (uma linha por anotação)
generate_sql_inserts(
    input_csv=str(BASE / "mlcq_enriched_dataset.csv"),
    output_sql=str(BASE / "mlcq_smell_occurrences_inserts.sql"),
)

# Inserts para mlcq_files (uma linha por arquivo)
generate_sql_inserts_for_files(
    input_csv=str(BASE / "mlcq_files.csv"),
    output_sql=str(BASE / "mlcq_files_inserts.sql"),
)

SQL file generated: /home/marcelo/Doutorado/llm-code-review-on-smells/datasets/mlcq_smell_occurrences_inserts.sql
SQL file generated: /home/marcelo/Doutorado/llm-code-review-on-smells/datasets/mlcq_files_inserts.sql


'/home/marcelo/Doutorado/llm-code-review-on-smells/datasets/mlcq_files_inserts.sql'

## 6. Aplicar dados no DB

Lê `mlcq_enriched_dataset.csv` (filtra TARGET_SMELLS + RELEVANT_SEVERITIES) e `mlcq_files.csv`, e faz batch insert via `psycopg2.extras.execute_values`. Idempotente: `ON CONFLICT DO NOTHING` em ambas as tabelas, então re-rodar não duplica.

In [47]:
import sys
import ast
import psycopg2
from psycopg2.extras import execute_values

_project_root = next(
    p for p in [BASE.parent, *BASE.parents]
    if (p / "managers").exists() and (p / "start_here").exists()
)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from start_here.experiments.constants import (
    RESULTS_DATABASE_NAME,
    RESULTS_DATABASE_HOST,
    RESULTS_DATABASE_PORT,
    RESULTS_DATABASE_USER,
    RESULTS_DATABASE_PASSWORD,
)

TARGET_SMELLS = ["feature envy", "long method", "data class"]
RELEVANT_SEVERITIES = ["major", "critical"]
SEVERITY_PRIORITY = {"critical": 0, "major": 1}


def _none_if_nan(value):
    return None if pd.isna(value) else value


def _parse_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    return ast.literal_eval(str(value))


occurrences = pd.read_csv(BASE / "mlcq_enriched_dataset.csv")
occurrences = occurrences[
    occurrences["smell"].isin(TARGET_SMELLS)
    & occurrences["severity"].isin(RELEVANT_SEVERITIES)
].copy()
# Dedup: 1 anotação por (repo, commit, path, smell). Critical > major em caso de divergência
occurrences = occurrences.assign(_sev_rank=occurrences["severity"].map(SEVERITY_PRIORITY))
occurrences = (
    occurrences.sort_values(["_sev_rank", "id"])
    .drop_duplicates(subset=["repo_url", "commit_hash", "file_path", "smell"], keep="first")
    .drop(columns="_sev_rank")
)

files_df = pd.read_csv(BASE / "mlcq_files.csv", sep=";")

occurrence_rows = [
    (
        int(row["id"]),
        row["repo_url"],
        row["commit_hash"],
        row["file_path"],
        int(row["start_line"]),
        int(row["end_line"]),
        _none_if_nan(row["code_snippet"]),
        _none_if_nan(row["file_content"]),
        row["smell"],
        row["severity"],
        _none_if_nan(row["type"]),
        _none_if_nan(row["code_name"]),
    )
    for _, row in occurrences.iterrows()
]

file_rows = [
    (
        row["repo_url"],
        row["commit_hash"],
        row["file_path"],
        _none_if_nan(row["file_content"]),
        _parse_list(row["annotated_smells"]),
        int(row["annotation_count"]),
        _parse_list(row["smell_occurrence_ids"]),
    )
    for _, row in files_df.iterrows()
]

connection = psycopg2.connect(
    dbname=RESULTS_DATABASE_NAME,
    user=RESULTS_DATABASE_USER,
    password=RESULTS_DATABASE_PASSWORD,
    host=RESULTS_DATABASE_HOST,
    port=RESULTS_DATABASE_PORT,
)

try:
    with connection.cursor() as cursor:
        execute_values(
            cursor,
            "INSERT INTO mlcq_smell_occurrences "
            "(id, repo_url, commit_hash, file_path, start_line, end_line, "
            "code_snippet, file_content, smell, severity, type, code_name) "
            "VALUES %s ON CONFLICT (id) DO NOTHING",
            occurrence_rows,
            page_size=100,
        )
    connection.commit()
    print(f"mlcq_smell_occurrences: {len(occurrence_rows)} rows enviadas")

    with connection.cursor() as cursor:
        execute_values(
            cursor,
            "INSERT INTO mlcq_files "
            "(repo_url, commit_hash, file_path, file_content, "
            "annotated_smells, annotation_count, smell_occurrence_ids) "
            "VALUES %s ON CONFLICT (repo_url, commit_hash, file_path) DO NOTHING",
            file_rows,
            page_size=100,
        )
    connection.commit()
    print(f"mlcq_files: {len(file_rows)} rows enviadas")

    with connection.cursor() as cursor:
        cursor.execute("SELECT COUNT(*) FROM mlcq_smell_occurrences")
        print(f"mlcq_smell_occurrences total no DB: {cursor.fetchone()[0]}")
        cursor.execute("SELECT COUNT(*) FROM mlcq_files")
        print(f"mlcq_files total no DB:             {cursor.fetchone()[0]}")
finally:
    connection.close()

mlcq_smell_occurrences: 476 rows enviadas
mlcq_files: 436 rows enviadas
mlcq_smell_occurrences total no DB: 476
mlcq_files total no DB:             436


In [26]:
import psycopg2
from start_here.experiments.constants import (
    RESULTS_DATABASE_NAME, RESULTS_DATABASE_HOST, RESULTS_DATABASE_PORT,
    RESULTS_DATABASE_USER, RESULTS_DATABASE_PASSWORD,
)
conn = psycopg2.connect(dbname=RESULTS_DATABASE_NAME, user=RESULTS_DATABASE_USER,
                        password=RESULTS_DATABASE_PASSWORD, host=RESULTS_DATABASE_HOST,
                        port=RESULTS_DATABASE_PORT)
with conn.cursor() as c:
    c.execute("SELECT COUNT(*) FROM mlcq_smell_occurrences WHERE severity NOT IN ('major','critical')")
    print("[DB] smell_occurrences fora do filtro:", c.fetchone()[0])
    c.execute("SELECT COUNT(*) FROM mlcq_smell_occurrences")
    print("[DB] mlcq_smell_occurrences total:", c.fetchone()[0])
    c.execute("SELECT COUNT(*) FROM mlcq_files")
    print("[DB] mlcq_files total:", c.fetchone()[0])

    c.execute("""
        SELECT f.id, f.smell_occurrence_ids,
               ARRAY(SELECT occ_id FROM unnest(f.smell_occurrence_ids) occ_id
                     WHERE NOT EXISTS (SELECT 1 FROM mlcq_smell_occurrences WHERE id = occ_id)) AS ids_sem_match,
               ARRAY(SELECT occ_id FROM unnest(f.smell_occurrence_ids) occ_id
                     JOIN mlcq_smell_occurrences o ON o.id = occ_id
                     WHERE o.severity NOT IN ('major','critical')) AS ids_apontando_none
        FROM mlcq_files f
        WHERE EXISTS (SELECT 1 FROM unnest(f.smell_occurrence_ids) occ_id
                      WHERE NOT EXISTS (SELECT 1 FROM mlcq_smell_occurrences WHERE id = occ_id)
                      OR EXISTS (SELECT 1 FROM mlcq_smell_occurrences WHERE id = occ_id AND severity NOT IN ('major','critical')))
        LIMIT 5
    """)
    for r in c.fetchall(): print("[DB]", r)
conn.close()


[DB] smell_occurrences fora do filtro: 0
[DB] mlcq_smell_occurrences total: 809
[DB] mlcq_files total: 436


In [27]:
import ast

files_df = pd.read_csv(BASE / "mlcq_files.csv", sep=";")
print(f"[CSV] mlcq_files.csv shape: {files_df.shape}")

all_ids_in_csv = set()
for s in files_df["smell_occurrence_ids"]:
    for i in ast.literal_eval(str(s)):
        all_ids_in_csv.add(int(i))
print(f"[CSV] ids únicos em mlcq_files.csv: {len(all_ids_in_csv)}")

enriched = pd.read_csv(BASE / "mlcq_enriched_dataset.csv")
filtered = enriched[
    enriched["smell"].isin(["feature envy", "long method", "data class"])
    & enriched["severity"].isin(["major", "critical"])
]
print(f"[CSV] total enriched: {len(enriched)}; filtrado major/critical+TARGET_SMELLS: {len(filtered)}")

filtered_ids = set(filtered["id"].astype(int).tolist())
extras_no_csv = all_ids_in_csv - filtered_ids
print(f"[CSV] ids no mlcq_files.csv que NÃO estão no enriched filtrado: {len(extras_no_csv)}")
if extras_no_csv:
    print(f"[CSV] primeiros 5: {sorted(extras_no_csv)[:5]}")
    bad = enriched[enriched["id"].isin(list(extras_no_csv)[:5])][["id","smell","severity"]]
    print("[CSV] essas annotações no enriched original:")
    print(bad)


[CSV] mlcq_files.csv shape: (436, 7)
[CSV] ids únicos em mlcq_files.csv: 809
[CSV] total enriched: 11603; filtrado major/critical+TARGET_SMELLS: 809
[CSV] ids no mlcq_files.csv que NÃO estão no enriched filtrado: 0


In [40]:
import psycopg2
from start_here.experiments.constants import (
    RESULTS_DATABASE_NAME, RESULTS_DATABASE_HOST, RESULTS_DATABASE_PORT,
    RESULTS_DATABASE_USER, RESULTS_DATABASE_PASSWORD,
)

conn = psycopg2.connect(
    dbname=RESULTS_DATABASE_NAME, user=RESULTS_DATABASE_USER,
    password=RESULTS_DATABASE_PASSWORD, host=RESULTS_DATABASE_HOST,
    port=RESULTS_DATABASE_PORT,
)
try:
    with conn.cursor() as c:
        c.execute("TRUNCATE mlcq_smell_occurrences, mlcq_files RESTART IDENTITY")
    conn.commit()
    print("tabelas truncadas")
finally:
    conn.close()


tabelas truncadas
